In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.decomposition import LatentDirichletAllocation

poi_matrix = np.zeros((40000, 85), dtype=np.float32)
directory = "cell_POIcat.csv"

for filename in os.listdir(directory):
    parts = filename.split(',')
    if len(parts) != 4:
        continue
    try:
        x, y, cat_id, count = map(int, parts)
        grid_idx = (x - 1) * 200 + (y - 1)
        poi_matrix[grid_idx, cat_id - 1] = count
    except ValueError:
        continue

poi_df = pd.DataFrame(poi_matrix)
print(f"Non-zero cells: {(poi_matrix.sum(axis=1) > 0).sum():,} / 40,000")

POI matrix shape: (40000, 85)
Non-zero cells: 20,146 / 40,000


In [ ]:
import ast, re

# Load the POI categories CSV
# Columns: x, y, top11_venues, zone_label, top5_categories
poi_cats = pd.read_csv(
    "POI_datacategories_with_top11.csv",
    header=None,
    names=["x", "y", "top_venues", "zone_label", "top_categories"]
)

# The top5_categories column looks like:
# "Business and Professional Services(12), Retail(5), Dining and Drinking(4), ..."
# Parse it into a dict of {category_name: count}
def parse_category_counts(raw):
    if pd.isna(raw):
        return {}
    result = {}
    for match in re.finditer(r'([^,(]+)\((\d+)\)', str(raw)):
        cat_name = match.group(1).strip()
        count = int(match.group(2))
        result[cat_name] = result.get(cat_name, 0) + count
    return result

poi_cats['cat_dict'] = poi_cats['top_categories'].apply(parse_category_counts)

# Get all unique top-level category names across the dataset
all_top_cats = sorted(set(
    cat for d in poi_cats['cat_dict'] for cat in d.keys()
))
print(f"Unique top-level categories found: {len(all_top_cats)}")
print(all_top_cats)

In [2]:
lda = LatentDirichletAllocation(
    n_components=5,
    random_state=42,
    learning_method='batch',
    max_iter=20
)
zone_distributions = lda.fit_transform(poi_df)  

zone_cols = [f'zone_{i}_prob' for i in range(5)]
dist_df = pd.DataFrame(zone_distributions, columns=zone_cols)
poi_df = pd.concat([poi_df, dist_df], axis=1)
poi_df['grid_id'] = range(40000)
poi_df['dominant_zone'] = dist_df.values.argmax(axis=1)

# Inspect what each zone learned
feature_names = [f"Cat_{i}" for i in range(1, 86)]
for topic_idx, topic in enumerate(lda.components_):
    top_features = [feature_names[i] for i in topic.argsort()[:-6:-1]]
    print(f"Zone {topic_idx} Top POIs: {top_features}")

print(f"\nZone distribution:\n{poi_df['dominant_zone'].value_counts().sort_index()}")

Zone 0 Top POIs: ['Cat_59', 'Cat_48', 'Cat_76', 'Cat_69', 'Cat_79']
Zone 1 Top POIs: ['Cat_4', 'Cat_40', 'Cat_48', 'Cat_13', 'Cat_69']
Zone 2 Top POIs: ['Cat_69', 'Cat_62', 'Cat_51', 'Cat_66', 'Cat_63']
Zone 3 Top POIs: ['Cat_81', 'Cat_84', 'Cat_75', 'Cat_63', 'Cat_79']
Zone 4 Top POIs: ['Cat_74', 'Cat_60', 'Cat_73', 'Cat_47', 'Cat_79']

Zone distribution:
dominant_zone
0    22871
1     1526
2     5049
3     4235
4     6319
Name: count, dtype: int64


In [3]:
zone_features = poi_df[['grid_id', 'dominant_zone'] + zone_cols]
zone_features.to_parquet("grid_zone_features2.parquet", index=False)
print("Saved grid_zone_features2.parquet")
print(zone_features.head())

Saved grid_zone_features2.parquet
   grid_id  dominant_zone  zone_0_prob  zone_1_prob  zone_2_prob  zone_3_prob  \
0        0              4     0.259197     0.012662     0.012762     0.012566   
1        1              2     0.029395     0.028969     0.531031     0.028890   
2        2              4     0.011954     0.011992     0.087271     0.012181   
3        3              4     0.018481     0.018398     0.018701     0.202769   
4        4              4     0.023068     0.022696     0.179092     0.022646   

   zone_4_prob  
0     0.702813  
1     0.381716  
2     0.876602  
3     0.741651  
4     0.752498  


In [ ]:
import os
import gzip

mobility_records = []
mob_file_csv = "yjmob100k-dataset1.csv"
mob_file_gz = "yjmob100k-dataset1.csv.gz"

if os.path.exists(mob_file_csv):
    open_func = open
    mob_file = mob_file_csv
elif os.path.exists(mob_file_gz):
    open_func = gzip.open
    mob_file = mob_file_gz
else:
    raise FileNotFoundError("Neither yjmob100k-dataset1.csv nor yjmob100k-dataset1.csv.gz found.")

with open_func(mob_file, 'rt') as f:
    for line in f:
        parts = line.strip().split(',')
        if len(parts) != 5:
            continue
        try:
            uid, day, x, y, time_slot = map(int, parts)
            grid_id = (x - 1) * 200 + (y - 1) 
            mobility_records.append({
                'uid': uid,
                'day': day,
                'x': x,
                'y': y,
                'time_slot': time_slot,
                'grid_id': grid_id
            })
        except ValueError:
            continue

mob_df = pd.DataFrame(mobility_records)
mob_df = mob_df.sort_values(['uid', 'day', 'time_slot']).reset_index(drop=True)

# Join zone features
zone_features = poi_df[['grid_id', 'dominant_zone'] + zone_cols]
mob_df = mob_df.merge(zone_features, on='grid_id', how='left')

print(f"Total mobility events: {len(mob_df):,}")
print(f"Unique users: {mob_df['uid'].nunique():,}")
print(f"Zone NaN rate: {mob_df['dominant_zone'].isna().mean():.2%}")
print(f"Day range: {mob_df['day'].min()} – {mob_df['day'].max()}")
print(f"Time slot range: {mob_df['time_slot'].min()} – {mob_df['time_slot'].max()}")
print(f"X range: {mob_df['x'].min()} – {mob_df['x'].max()}")
print(f"Y range: {mob_df['y'].min()} – {mob_df['y'].max()}")
print(mob_df.head(10))

In [5]:
# HuMob 2023 dataset starts on a Monday based on the paper
# Day 0 = Monday, so day % 7 gives day of week
mob_df['day_of_week'] = mob_df['day'] % 7       # 0=Mon ... 6=Sun
mob_df['is_weekend'] = (mob_df['day_of_week'] >= 5).astype(int)

# Time slot is 30-min intervals: slot 0 = midnight, slot 47 = 11:30pm
# Values above 47 in your data suggest it may be absolute — verify this
# If time_slot is already intra-day (0-47), use directly
# If absolute across full study, convert:
SLOTS_PER_DAY = 48
mob_df['intra_day_slot'] = mob_df['time_slot'] % SLOTS_PER_DAY
mob_df['hour'] = (mob_df['intra_day_slot'] * 30) // 60

# Rough period labels useful for debugging/inspection
def label_period(h):
    if 6 <= h < 10:   return 'morning_commute'
    elif 10 <= h < 17: return 'daytime'
    elif 17 <= h < 21: return 'evening_commute'
    elif 21 <= h < 24: return 'night'
    else:              return 'late_night'

mob_df['period'] = mob_df['hour'].apply(label_period)

print(mob_df[['uid','day','day_of_week','is_weekend','time_slot','intra_day_slot','hour','period']].head(15))

    uid  day  day_of_week  is_weekend  time_slot  intra_day_slot  hour  \
0     0    0            0           0         82              34    17   
1     0    0            0           0         83              35    17   
2     0    0            0           0         84              36    18   
3     0    0            0           0         85              37    18   
4     0    0            0           0         86              38    19   
5     0    0            0           0         86              38    19   
6     0    0            0           0         86              38    19   
7     0    0            0           0         86              38    19   
8     0    0            0           0         86              38    19   
9     0    0            0           0         86              38    19   
10    0    0            0           0         86              38    19   
11    0    0            0           0         88              40    20   
12    0    0            0           0 

In [ ]:
zone_features = pd.read_parquet("grid_zone_features2.parquet")
mob_df = mob_df.merge(zone_features, on='grid_id', how='left')

nan_rate = mob_df['dominant_zone_x'].isna().mean()
print(f"Zone NaN rate: {nan_rate:.2%}")

# Fill unmatched cells with a neutral fallback zone token
# In LP-BERT vocabulary this will map to a dedicated UNKNOWN_ZONE token
mob_df['dominant_zone'] = mob_df['dominant_zone_x'].fillna(-1).astype(int)

print(f"\nDominant zone distribution:")
print(mob_df['dominant_zone'].value_counts().sort_index())
print(mob_df[['uid','day','x','y','grid_id','dominant_zone','is_weekend']].head(10))

KeyError: 'dominant_zone'

In [7]:
# Debugging: Check columns after loading and merging
print("zone_features columns:", zone_features.columns.tolist())
print("mob_df columns after merge:", mob_df.columns.tolist())
print("Sample of zone_features:")
print(zone_features.head())
print("Sample of mob_df after merge:")
print(mob_df.head())

zone_features columns: ['grid_id', 'dominant_zone', 'zone_0_prob', 'zone_1_prob', 'zone_2_prob', 'zone_3_prob', 'zone_4_prob']
mob_df columns after merge: ['uid', 'day', 'x', 'y', 'time_slot', 'grid_id', 'dominant_zone_x', 'zone_0_prob_x', 'zone_1_prob_x', 'zone_2_prob_x', 'zone_3_prob_x', 'zone_4_prob_x', 'day_of_week', 'is_weekend', 'intra_day_slot', 'hour', 'period', 'dominant_zone_y', 'zone_0_prob_y', 'zone_1_prob_y', 'zone_2_prob_y', 'zone_3_prob_y', 'zone_4_prob_y']
Sample of zone_features:
   grid_id  dominant_zone  zone_0_prob  zone_1_prob  zone_2_prob  zone_3_prob  \
0        0              4     0.259197     0.012662     0.012762     0.012566   
1        1              2     0.029395     0.028969     0.531031     0.028890   
2        2              4     0.011954     0.011992     0.087271     0.012181   
3        3              4     0.018481     0.018398     0.018701     0.202769   
4        4              4     0.023068     0.022696     0.179092     0.022646   

   zone_4_p

In [ ]:
# Each user's full history becomes one sequence
# Each event is a tuple of (grid_id, dominant_zone, is_weekend, intra_day_slot)
# This is the enriched token that replaces raw grid coordinates in LP-BERT

sequences = []
for uid, group in mob_df.groupby('uid'):
    group = group.sort_values(['day', 'intra_day_slot'])
    seq = group[['grid_id', 'dominant_zone', 'is_weekend', 'intra_day_slot']].values.tolist()
    sequences.append({
        'uid': uid,
        'sequence': seq,
        'seq_len': len(seq)
    })

seq_df = pd.DataFrame(sequences)
seq_df.to_parquet("user_sequences.parquet", index=False)

print(f"Total users: {len(seq_df):,}")
print(f"Avg sequence length: {seq_df['seq_len'].mean():.1f}")
print(f"Min/Max: {seq_df['seq_len'].min()} / {seq_df['seq_len'].max()}")
print("\nSample sequence (first 5 events for user 0):")
print(seq_df.iloc[0]['sequence'][:5])

In [ ]:
# Task 1: Days 0-59 are training, Days 60-74 are to be predicted
TRAIN_CUTOFF = 60

train_df = mob_df[mob_df['day'] < TRAIN_CUTOFF].copy()
test_df  = mob_df[mob_df['day'] >= TRAIN_CUTOFF].copy()

print(f"Training events: {len(train_df):,}  (Days 0–59)")
print(f"Test events:     {len(test_df):,}   (Days 60–74)")
print(f"Training users:  {train_df['uid'].nunique():,}")
print(f"Test users:      {test_df['uid'].nunique():,}")

train_df.to_parquet("train_sequences.parquet", index=False)
test_df.to_parquet("test_sequences.parquet",  index=False)